In [9]:
import os
import time
import getpass
from langchain_cohere import ChatCohere
from langchain.chat_models import init_chat_model

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass.getpass("Enter Cohere API key:")

llm = init_chat_model("command-a-03-2025", model_provider="cohere", temperature=0)

In [42]:
from typing import List, Optional, TypedDict
from pydantic import BaseModel, Field

class SchemaExtraction(BaseModel):
    """A schema for extracting entities from text."""
    #  Trich xuất các thông tin đến người dùng
    name: Optional[str] = Field(..., description="Tên của người đặt câu hỏi")
    age_group: Optional[str] = Field(...,
                                     description="Nhóm tuổi của người đặt câu hỏi hoặc có thể suy ra",
                                     enum=["child", "teen", "adult", "senior"])
    language:Optional[str] = Field(...,
                                   description="Ngôn ngữ được dùng để đặt câu hỏi.",
                                   enum=["vietnamese", "english", "french", "chinese"])
    level: Optional[str] = Field(...,
                                 description="Mức độ (level) hiểu biết về lịch sử của người đặt câu hỏi (beginner: thấp, intermediate: vừa, advanced: cao)",
                                 enum=["beginner", "intermediate", "advanced"],)
    tone_preference: Optional[str] = Field(..., description="Phong cách phản hồi người dùng ưa thích, ví dụ: hài hước, nghiêm túc, thân mật, học thuật.")
    current_emotion: Optional[str] = Field(..., description="Cảm xúc hiện tại nếu có thể suy luận được, ví dụ: tò mò, buồn, vui, thất vọng.")
    
    # Trích xuất các thông tin đến câu hỏi
    current_topic: Optional[str] = Field(..., description="Chủ đề hiện tại mà người dùng đang quan tâm.")
    interested_characters: Optional[List[str]] = Field(..., description="Các nhân vật lịch sủ yêu thích của người dùng.")
    emotional_expression: Optional[str] = Field(..., description="Cảm xúc hiện tại của người dùng. Ví dụ: Eo nghe sợ vậy, Ác quá trời, Thật thú vị...")

    # Các câu hỏi không rõ ràng
    relative_question: Optional[str] = Field(..., description="Các câu hỏi có vẻ như liên quan đến chủ đề thảo luận trước đó. Ví dụ: Lúc nãy bạn nói..., Bạn có thể nói rõ hơn về vấn đề đó được không?...")
    keywords: Optional[List[str]] = Field(..., description="Các keywords quan trọng trong câu hỏi chính hoặc chủ đề cần lưu ý.")
    question_summary: Optional[str] = Field(..., description="Hãy tóm tắt lại câu hỏi")

In [43]:
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.prompts import ChatMessagePromptTemplate

prompt_template = ChatPromptTemplate.from_messages([
    ("system", """Bạn là một chuyên gia trong lĩnh vực trích xuất thông tin liên quan tới người dùng. 
     Hãy trích xuất các thông tin như tên và tuổi trong văn bản được cung cấp. Nếu không biết, hãy trả về giá trị null"""),
    ("human", "{text}"),
])

In [44]:
text = """
Xin chào! Mình tên là Úc. Mình đã trải qua 25 nồi bánh chưng, một nồi bánh tương ứng với 1 năm. 
Mình là một chuyên gia hỏi đáp lịch sử và hôm nay muốn trao đổi về cuộc chiến Điện Biên Phủ.
Bạn nghĩ thế nào về cuộc chiến này, bạn có thể so sánh cuộc chiến này với cuộc chiến được đề cập ở trên không?
"""
structured_output = llm.with_structured_output(schema=SchemaExtraction)
prompt = prompt_template.invoke({"text": text})
output = structured_output.invoke(prompt)

print(output)
for field, value in output.model_dump().items():
    print(f"{field}: {value}")

name='Úc' age_group='adult' language='vietnamese' level='advanced' tone_preference='friendly' current_emotion='excited' current_topic='Chiến dịch Điện Biên Phủ' interested_characters=None emotional_expression=None relative_question='Bạn nghĩ thế nào về cuộc chiến này, bạn có thể so sánh cuộc chiến này với cuộc chiến được đề cập ở trên không?' keywords=['Chiến dịch Điện Biên Phủ', 'lịch sử', 'chiến tranh'] question_summary='Bạn nghĩ thế nào về cuộc chiến này, bạn có thể so sánh cuộc chiến này với cuộc chiến được đề cập ở trên không?'
name: Úc
age_group: adult
language: vietnamese
level: advanced
tone_preference: friendly
current_emotion: excited
current_topic: Chiến dịch Điện Biên Phủ
interested_characters: None
emotional_expression: None
relative_question: Bạn nghĩ thế nào về cuộc chiến này, bạn có thể so sánh cuộc chiến này với cuộc chiến được đề cập ở trên không?
keywords: ['Chiến dịch Điện Biên Phủ', 'lịch sử', 'chiến tranh']
question_summary: Bạn nghĩ thế nào về cuộc chiến này, bạn